# Scale hints: VLM side (Kaggle 2xT4, or Colab)

Kaggle: Settings -> Accelerator **GPU T4 x2**, Internet **on**. Colab: Runtime -> T4 (or better).

Run cells top to bottom. Every GPU step resumes, so if the session dies just rerun from the setup cell.
MapAnything (teacher) runs in `notebooks/teacher.ipynb` on a **separate** machine/session in parallel; commit its `cache/teacher_scale.json` to the repo, then run the `git pull` cell below.

In [ ]:
import os, sys, subprocess
REPO_URL = 'https://github.com/YOUR_USER/gsrvlm.git'   # <- edit
ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB = 'google.colab' in sys.modules
WORK = '/kaggle/working' if ON_KAGGLE else '/content'
if ON_COLAB:
    # keep results on Drive so a Colab disconnect loses nothing
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive'
os.chdir(WORK)
if not os.path.exists('gsrvlm'):
    subprocess.run(['git', 'clone', REPO_URL, 'gsrvlm'], check=True)
os.chdir('gsrvlm')
print(os.getcwd())
!git pull
!nvidia-smi -L

In [ ]:
!pip install -q -r requirements.txt
import transformers, torch; print('transformers', transformers.__version__, '| torch', torch.__version__)

## Data: questions, split, 16 frames per scene (CPU, ~20-40 min, mostly download)
Zips go to local `/tmp` and are deleted after frames are extracted.

In [ ]:
!python prepare_data.py --download_dir /tmp/vsi
!ls cache/frames | wc -l

## Smoke test (2 min): 5 questions per task, check answers parse

In [ ]:
!python run_vlm.py --condition baseline --split eval --limit_per_task 5 --out_dir results_smoke
!python -c "import json; [print(r['task'][:12], r['gt'], '|', r['raw_answer'][:60], '->', r['pred'], round(r['mra'],2)) for r in map(json.loads, open('results_smoke/baseline.s0.jsonl'))]"

## E0/E1: baseline on ALL scenes, then E2 decomposition -> GO / NO-GO
If NO-GO (or the readout-collapse warning prints), stop and look at the numbers before spending GPU on hints.

In [ ]:
!SPLIT=all bash run_conds.sh baseline
!python decompose.py --n_boot 200

## E4a: hints that need no teacher (run while the teacher runs elsewhere)
`spatialprompt` reasons at length (512 tokens), so it is capped at 100 questions per task.

In [ ]:
!bash run_conds.sh placebo anchor anchor_k0.5 anchor_k2 anchor_k0.71 anchor_k1.41
!EXTRA='--limit_per_task 100' bash run_conds.sh spatialprompt

## E4b: teacher hints (after `cache/teacher_scale.json` is committed to the repo)

In [ ]:
!git pull
!python -c "import json; T=json.load(open('cache/teacher_scale.json')); print(len(T), 'scenes,', sum(1 for v in T.values() if v.get('s_med')), 'ok')"
!bash run_conds.sh teacher shuffled teacher_k0.5 teacher_k2 teacher_k0.71 teacher_k1.41
!EXTRA='--tasks room_size_estimation' bash run_conds.sh teacher_ext

## E6: vision-token cache -> predictor -> `learned` hint

In [ ]:
import torch
os.makedirs('logs', exist_ok=True)
n = torch.cuda.device_count()
procs = [subprocess.Popen(f'CUDA_VISIBLE_DEVICES={g} python cache_vis.py --shard {g} --nshards {n} > logs/cache_vis.s{g}.log 2>&1', shell=True) for g in range(n)]
print([p.wait() for p in procs])
!tail -n 3 logs/cache_vis.s*.log
!python train_predictor.py
!bash run_conds.sh learned

## Analysis + bundle results for download

In [ ]:
!python decompose.py --n_boot 1000
!python elasticity.py
!python make_tables.py
!zip -qr results_bundle.zip results figures cache/teacher_scale.json cache/learned_scale.json splits data/questions.jsonl logs
print('download results_bundle.zip from the output / files pane')